# HW2: Teach an LLM to Write Checks with LoRA

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/HW2-LoRA-Check-Writer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers peft trl
!pip uninstall -q -y torchao   # the old torchao preinstalled on Colab is incompatible with recent PEFT
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['zh_numbers.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In `05-LoRA-SFT` we taught the model to convert Chinese numerals into Arabic digits. This time it goes the other way:

**convert Arabic numbers into the formal Chinese numerals written on checks**, e.g. `3052` → `參仟零伍拾貳`, `10050` → `壹萬零伍拾`.

Formal ("banker's") numerals use complex characters — 壹 貳 參 肆 伍 陸 柒 捌 玖 for 1–9, 拾 佰 仟 萬 for ten / hundred / thousand / ten-thousand — so the amount on a check cannot easily be altered. As with ordinary Chinese numerals, numbers are grouped by 萬 (10⁴), and a run of missing places inside a number is written as a single 零 (zero).

This is harder than last time: the model must put 零 and 萬 in exactly the right places.

Rules
* The base model must be `Qwen/Qwen3-0.6B` (you may use LoRA or other PEFT methods, or full fine-tuning)
* You can generate as much training data as you like with `zh_numbers.py`
* Testing uses greedy decoding (`do_sample=False`), and an answer counts only if it is **exactly identical**
* Must run on a Colab T4, with total training time under 30 minutes
* Submit: the notebook (with outputs)

Directions to try
* Amount of data and the distribution of numbers (more numbers containing 零?)
* LoRA `r`, `target_modules`, learning rate, epochs
* Prompt design
* Let the model "think" first (e.g. split the number into ten-thousands, thousands, hundreds, tens) before answering?

In [ ]:
import random
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from zh_numbers import num2zh_upper, random_number
device = "cuda" if torch.cuda.is_available() else "cpu"
for n in [5, 10, 105, 3052, 10050, 10000005, 10100100]:
    print(n, num2zh_upper(n))

## Test data (do not modify)

In [ ]:
test_rng = random.Random(2026)
test_numbers = [random_number(test_rng) for _ in range(300)]
test_answers = [num2zh_upper(n) for n in test_numbers]

## Your code

Complete `predict(numbers)`: it takes a list of ints and returns a list of the strings the model generates.

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

PROMPT = "Convert the following number to formal Chinese numerals (as written on checks). Output only the answer: {}"

# TODO: generate training data and train with SFTTrainer + LoRA (see 05-LoRA-SFT)

@torch.no_grad()
def predict(numbers, batch_size=50):
    tokenizer.padding_side = "left"
    model.eval()
    results = []
    for i in range(0, len(numbers), batch_size):
        texts = [tokenizer.apply_chat_template([{"role": "user", "content": PROMPT.format(n)}], tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
                 for n in numbers[i:i + batch_size]]
        inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        results += [tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]
    return results

## Grading (do not modify)

In [ ]:
preds = predict(test_numbers)
acc = sum(p == a for p, a in zip(preds, test_answers)) / len(test_answers)
for n, p, a in list(zip(test_numbers, preds, test_answers))[:10]:
    print(f"{n:>10} {p}  {'✅' if p == a else '❌ ' + a}")
print(f"test accuracy = {acc:.4f}")

## Scoring (40 points)

* 98% or higher: 40 points
* 95%: 35 points
* 90%: 28 points
* 80%: 20 points
* 50%: 10 points

(Scores in between are linearly interpolated.)